# Lexical of 20th century Odyssey translations (Part B): Etymologies


____________________________________________________
## **Road Map**

**I. Libraries, files, and paths**

**II. The Texts**

1. Bibliographic information about the translators  
2. The translators at a glance tokenwise   

**III. TTR Analysis**

1. All-in, straightforward model  
    a) TTR Computation  
    b) Shapiro-Wilk test to check for normality  
    c) One-wat ANOVA for overall differences  
    d) Pairwise t-test using Bonferroni coprrection  
    c) Meassuring effect size ussing Cohen's d  

2. Adaptive models  
    a) Mixed-Effects model: author fixed effect / book as random effect  
    b) Standardized TTR:   
    c) Moving-average TTR: translation as temporal change  

3. Supplement models  
    a) Lexical Density   
    b) Diachronic analysis  
    c) Semantic fields:  

**IV. Zipf's Law**

**V. TF-IDF**



**VI. Discussing Results**

In [36]:
import autotime # Provision for anxious people
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 3.59 ms (started: 2025-04-21 15:39:43 +02:00)


In [37]:
# ----------------------------------------------------------------------
# Baic Libraries
# ----------------------------------------------------------------------

import sys 
import os

import ast
from collections import Counter

import re
import nltk

import numpy as np
import pandas as pd

import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

import scipy.stats as stats
from itertools import combinations

time: 696 μs (started: 2025-04-21 15:39:43 +02:00)


In [38]:
# ----------------------------------------------------------------------
# Personalized Visualization & Functions
# ---------------------------------------------------------------------- 

sys.path.append('/Users/debr/English-Homer/functions') 
import matplotlib.pyplot as plt
import seaborn as sns

import e_chroma as chroma # My Vizualization library
import e_plots as oz      # My custom plots library
import e_pandisplay as pan# My pandas display options

import e_nlp_ody as e     # Import my nlp functions

import warnings           # Nononsense provision
warnings.filterwarnings('ignore')

time: 2.96 ms (started: 2025-04-21 15:39:43 +02:00)


In [39]:
# ----------------------------------------------------------------------
# File management
# ----------------------------------------------------------------------

# TO UPDATE
nb_id = "lexical_B01"

output_path = f"./"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
output_path_plots = f"./{output_path}/{nb_id}_plots/"
os.makedirs(os.path.dirname(output_path_plots), exist_ok=True)
chroma.set_output_path(output_path_plots)

Output path set to: ././/lexical_B01_plots/
time: 1.2 ms (started: 2025-04-21 15:39:43 +02:00)


In [40]:
# ----------------------------------------------------------------------
# Odysseys
# ----------------------------------------------------------------------

translators = ['AT_Murray', 'Fitzgerald', 'Lattimore', 'Fagles', 'Wilson', 'Green', 'Woolf']

dfs = []

for odyssey in translators:
    filepath = f"/Users/debr/odysseys_en/dataframed/Odyssey_{odyssey}_DataFrame.csv"
    temp_df = pd.read_csv(filepath)  
    dfs.append(temp_df)  # Append it to the list

df = pd.concat(dfs, axis=0, ignore_index=True)

df["text"] = df["text"].apply(ast.literal_eval)
df["tokens"] = df["tokens"].apply(ast.literal_eval)
df['translator'] = pd.Categorical(df['author'])
df["book_num"] = pd.Categorical(df["book_num"])
df = df[['translator', 'book_num', 'text', 'tokens', 'num_words', 'num_tokens']]

# ----------------------------------------------------------------------
# Backup dataframe only 'translator', 'book_num', 'text', 'tokens', columns
# ----------------------------------------------------------------------
df_bkp = df[['translator', 'book_num', 'text', 'tokens']].copy()
# ----------------------------------------------------------------------
# Dataframe check
# ----------------------------------------------------------------------

e.check_df(df)

Mr righteous here has no missing values!

* df columns: Index(['translator', 'book_num', 'text', 'tokens', 'num_words', 'num_tokens'], dtype='object') 

* Shape: (168, 6) 

* Total memory in MB: 4.064749
time: 691 ms (started: 2025-04-21 15:39:43 +02:00)


## **2. The Dictionaries

Python etymologies package is Ety. It is based on Melo's dictionary. However, it doesn't works as expected. So I had to came up with my own solution.

This is the data-dictionary. 

Etymological Wordnet 2013-02-08
Gerard de Melo
http://icsi.berkeley.edu/~demelo/etymwn/


== DESCRIPTION ==

The Etymological Wordnet project provides information about how words in different languages 
are etymologically related. The information is mostly mined from the English version of
Wiktionary, but also contains a number of manual additions.


== FORMAT ==

The package includes a Tab-separated values (TSV) file in UTF-8 format with three columns,
providing a word, a relation, and a target word. Words are given with ISO 639-3 codes
(additionally, there are some ISO 639-2 codes prefixed with "p_" to indicate proto-languages).
The most relevant relation is "rel:etymology". To see only etymological relations, run
  grep "rel:etymology" etymwn.tsv | less
on UNIX-based systems.


== CREDITS AND LICENSE ==

Gerard de Melo
http://icsi.berkeley.edy/~demelo/
Based on the contributions of the English Wiktionary community
http://en.wiktionary.org/

License: CC-BY-SA 3.0

In scientific works, please cite:
  Gerard de Melo, Gerhard Weikum. "Towards Universal Multilingual Knowledge Bases".
  In: Principles, Construction, and Applications of Multilingual Wordnets. Proceedings
  of the 5th Global Wordnet Conference (GWC 2010). Narosa Publishing 2010, New Delhi India.

### Working with ety

In [41]:
# ----------------------------------------------------------------------
# Probing ety
# ----------------------------------------------------------------------
import ety
# Get the etymology of the word "muse" in English
word = "muse"
ety_origin = ety.origins(word)
ety_tree = ety.tree(word)
print(f"Etymology of '{word}': {ety_origin}")
print(f"Etymology tree of '{word}': {ety_tree}")

Etymology of 'muse': [Word(muse, Middle French (ca. 1400-1600) [frm])]
Etymology tree of 'muse': muse (English)
└── muse (Middle French (ca. 1400-1600))
    └── Musa (Latin)
        └── Μοῦσα (Ancient Greek (to 1453))
time: 1.57 ms (started: 2025-04-21 15:39:44 +02:00)


Curiously, ety doesn't have the full etymology for "muse".

In [42]:
# Get the etymology of the word "table" in English

word = "table"
ety_origin = ety.origins(word)
ety_tree = ety.tree(word)
print(f"Etymology of '{word}': {ety_origin}")
print(f"Etymology tree of '{word}': {ety_tree}")

Etymology of 'table': [Word(table, Middle English (1100-1500) [enm])]
Etymology tree of 'table': table (English)
└── table (Middle English (1100-1500))
time: 851 μs (started: 2025-04-21 15:39:44 +02:00)


### Developing a dictionary we can use


In [43]:
# ----------------------------------------------------------------------
# Getting the full dictionary
# ----------------------------------------------------------------------

etymologypath = "/Users/debr/odysseys_en/etymwn-20130208/etymwn.tsv"

etymology_df = pd.read_csv(etymologypath, sep="\t", names=["word", "relation", "target_word"], encoding="utf-8")
etymology_df["relation"] = etymology_df["relation"].astype("category") # to save space

e.check_df(etymology_df)

Mr righteous here has no missing values!

* df columns: Index(['word', 'relation', 'target_word'], dtype='object') 

* Shape: (6031431, 3) 

* Total memory in MB: 922.924706
time: 4.45 s (started: 2025-04-21 15:39:44 +02:00)


In [44]:
etymology_df.sample(10, random_state=402)

,word,relation,target_word
5463534,spa: condicional,rel:has_derived_form,spa: condicionales
467142,eng: albuminization,rel:etymologically_related,eng: toxalbumin
2075742,fra: insulteur,rel:etymologically_related,fra: insulteuse
4897623,lat: torculo,rel:has_derived_form,lat: torculavissent
255143,deu: anfangen,rel:has_derived_form,deu: anfingt
4675182,lat: prodicimini,rel:is_derived_from,lat: prodico
1992445,fra: enflai,rel:is_derived_from,fra: enfler
5773738,spa: sobrecargasen,rel:is_derived_from,spa: sobrecargar
4882662,lat: taedeo,rel:has_derived_form,lat: taedueritis
5357602,spa: aclaraseis,rel:is_derived_from,spa: aclarar


time: 139 ms (started: 2025-04-21 15:39:48 +02:00)


In [45]:
relations = [etymology_df['relation'].unique()]
relations

[['rel:etymological_origin_of', 'rel:has_derived_form', 'rel:is_derived_from', 'rel:etymology', 'rel:etymologically_related', 'rel:variant:orthography', 'rel:derived', 'rel:etymologically']
 Categories (8, object): ['rel:derived', 'rel:etymological_origin_of', 'rel:etymologically', 'rel:etymologically_related', 'rel:etymology', 'rel:has_derived_form', 'rel:is_derived_from', 'rel:variant:orthography']]

time: 16.5 ms (started: 2025-04-21 15:39:48 +02:00)


**Description of categories**

| Relation                        | Description |
|----------------------------------|------------|
| **rel:etymological_origin_of**   | Indicates that a word is the root or ancestor of another word. Example: Latin *scientia* is the etymological origin of English *science*. |
| **rel:has_derived_form**         | Shows that a word has a derived variant. Example: *happy* has the derived form *happiness*. |
| **rel:is_derived_from**          | Specifies that a word originates from another word. This is the inverse of *rel:etymological_origin_of*. Example: English *science* is derived from Latin *scientia*. |
| **rel:etymology**                | A general relation indicating the etymology of a word without specifying a direction of derivation. |
| **rel:etymologically_related**   | Indicates that two words share a common etymology but are not directly derived from one another. Example: English *hospital* and *host* are etymologically related through Latin *hospes*. |
| **rel:variant:orthography**      | Refers to different spellings of the same word. Example: *color* (American English) vs. *colour* (British English). |
| **rel:derived**                  | A broad category that includes words that evolved from another language but does not specify the exact relationship. |
| **rel:etymologically**           | A general tag used to indicate some form of etymological connection. Often used when the exact type of relation is unclear. |

In [46]:
target_word = "enm: table"  # Change this to any word you want to match

etymology_root_1 = etymology_df[etymology_df["word"] == target_word]

print(etymology_root_1)

        word        relation                    target_word
1345511  enm: table  rel:etymological_origin_of  eng: table
time: 189 ms (started: 2025-04-21 15:39:48 +02:00)


In [47]:
# ----------------------------------------------------------------------
# A subset dictionary based on "relation"
# ----------------------------------------------------------------------

relation_etymology_df = etymology_df[etymology_df["relation"] == "rel:etymology"]
relation_etymology_df.sample(8, random_state=42)

,word,relation,target_word
1727960,fin: räkänokka,rel:etymology,fin: nokka
753310,eng: futureless,rel:etymology,eng: future
842271,eng: inveracity,rel:etymology,eng: in-
804350,eng: homœostasis,rel:etymology,eng: homœ-
1622981,fin: huumekauppias,rel:etymology,fin: huume
404060,eng: Ceylon,rel:etymology,grc: Σελεδίβα
1063714,eng: pritumumab,rel:etymology,eng: -tum-
5429877,spa: cabezota,rel:etymology,spa: -ota


time: 62.8 ms (started: 2025-04-21 15:39:49 +02:00)


In [48]:
# ----------------------------------------------------------------------
# Chasing the etymology tree
# ----------------------------------------------------------------------

target_word = "eng: ghost"
etymology_root_1 = relation_etymology_df[relation_etymology_df["word"] == target_word]
print("Root 1:", etymology_root_1)

target_word = "enm: gost" 
etymology_root_2 = relation_etymology_df[relation_etymology_df["word"] == target_word]
print("\nRoot 2:", etymology_root_2)

target_word = "ang: gast" # last branch registered
etymology_root_3 = relation_etymology_df[relation_etymology_df["word"] == target_word]
print("\nRoot 3:", etymology_root_3)

Root 1:        word        relation       target_word
761941  eng: ghost  rel:etymology  enm: gost 

Root 2:         word       relation       target_word
1342198  enm: gost  rel:etymology  ang: gast 

Root 3: Empty DataFrame
Columns: [word, relation, target_word]
Index: []
time: 70.9 ms (started: 2025-04-21 15:39:49 +02:00)


In [49]:
# ----------------------------------------------------------------------
# Function to get the etymology tree
# ----------------------------------------------------------------------

def trace_etymology(word, df):
    etymology_chain = [word]  # Store the lineage of words
    
    while True:
        # Find the row where 'word' matches
        etymology_row = df[df["word"] == word]
        
        # If no match is found, stop the loop
        if etymology_row.empty:
            break
        
        # Get the next word in the etymological chain
        next_word = etymology_row["target_word"].values[0]
        
        # Append to the chain and set up for the next iteration
        etymology_chain.append(next_word)
        word = next_word  # Set the next search target

    # Format output sentence
    if len(etymology_chain) > 1:
        etymology_str = ' → '.join(etymology_chain)
        print(f'The word "{etymology_chain[0]}" traces back through: {etymology_str}.')
    else:
        print(f'No etymological root found for "{word}".')

# Example usage
trace_etymology("eng: ghost", relation_etymology_df)

The word "eng: ghost" traces back through: eng: ghost → enm: gost → ang: gast.
time: 69.1 ms (started: 2025-04-21 15:39:49 +02:00)


In [50]:
# ----------------------------------------------------------------------
# The function in a loop for a list of words
# ----------------------------------------------------------------------

ety_inquiries = ['muse', 'ghost', 'table', 'sword', 'shield', 'spear', 'battle', 'warrior', 'hero', 'goddess']

for word in ety_inquiries:
    print(f"Word: {word}")
    trace_etymology(f"eng: {word}", relation_etymology_df)
    print()

Word: muse
The word "eng: muse" traces back through: eng: muse → frm: muse → lat: Musa → grc: Μοῦσα.

Word: ghost
The word "eng: ghost" traces back through: eng: ghost → enm: gost → ang: gast.

Word: table
The word "eng: table" traces back through: eng: table → enm: table.

Word: sword
The word "eng: sword" traces back through: eng: sword → enm: sword.

Word: shield
The word "eng: shield" traces back through: eng: shield → ang: scieldan.

Word: spear
The word "eng: spear" traces back through: eng: spear → ang: spere.

Word: battle
The word "eng: battle" traces back through: eng: battle → enm: batel → fro: bataille.

Word: warrior
No etymological root found for "eng: warrior".

Word: hero
No etymological root found for "eng: hero".

Word: goddess
The word "eng: goddess" traces back through: eng: goddess → eng: -ess → fra: -esse → lat: -issa → grc: -ισσα.

time: 617 ms (started: 2025-04-21 15:39:49 +02:00)


In [51]:
# ----------------------------------------------------------------------
# Deadends in the dictionary
# ----------------------------------------------------------------------

# Some words get lost between pre- and sufixes. E.g., "goddess"

word = "goddess"

# ----------------------------------------------------------------------
# ety module
# ----------------------------------------------------------------------

ety_origin = ety.origins(word)
ety_tree = ety.tree(word)
print(f"Etymology of '{word}': {ety_origin}")
print(f"Etymology tree of '{word}': {ety_tree}")

# ----------------------------------------------------------------------
# my function:
# ----------------------------------------------------------------------
print() 
trace_etymology(f"eng: {word}", relation_etymology_df)

Etymology of 'goddess': [Word(-ess, English [eng]), Word(god, English [eng])]
Etymology tree of 'goddess': goddess (English)
├── -ess (English)
│   └── -esse (French)
│       ├── -issa (Latin)
│       │   └── -ισσα (Ancient Greek (to 1453))
│       └── -itiam (Latin)
└── god (English)

The word "eng: goddess" traces back through: eng: goddess → eng: -ess → fra: -esse → lat: -issa → grc: -ισσα.
time: 115 ms (started: 2025-04-21 15:39:49 +02:00)


In [52]:
word = "table"

# ----------------------------------------------------------------------
# ety module
# ----------------------------------------------------------------------

ety_origin = ety.origins(word)
ety_tree = ety.tree(word)
print(f"Etymology of '{word}': {ety_origin}")
print(f"Etymology tree of '{word}': {ety_tree}")

# ----------------------------------------------------------------------
# my function:
# ----------------------------------------------------------------------
print() 
trace_etymology(f"eng: {word}", relation_etymology_df)

Etymology of 'table': [Word(table, Middle English (1100-1500) [enm])]
Etymology tree of 'table': table (English)
└── table (Middle English (1100-1500))

The word "eng: table" traces back through: eng: table → enm: table.
time: 46.5 ms (started: 2025-04-21 15:39:49 +02:00)


In [53]:
# ----------------------------------------------------------------------
# Illustrating deadends
# ----------------------------------------------------------------------

target_word = "eng: table"  
ety_deadend = "enm: table" # Middle English
etymology_root_1 = etymology_df[etymology_df["word"] == target_word]
etymology_root_2 = etymology_df[etymology_df["word"] == ety_deadend]
print("Slice of etymology database:")
print(etymology_root_1[24:30])
print("\nIllustrating deadend in etymology database:")
print(etymology_root_2)

Slice of etymology database:
        word        relation                    target_word             
1221570  eng: table  rel:etymologically_related              eng: tablet
1221571  eng: table  rel:etymologically_related            eng: tabulate
1221572  eng: table               rel:etymology               enm: table
1221573  eng: table        rel:has_derived_form        eng: Cayley table
1221574  eng: table        rel:has_derived_form      eng: billiard table
1221575  eng: table        rel:has_derived_form  eng: bring to the table

Illustrating deadend in etymology database:
        word        relation                    target_word
1345511  enm: table  rel:etymological_origin_of  eng: table
time: 354 ms (started: 2025-04-21 15:39:50 +02:00)


## **Designing our dictionary-look up**

### **Step 1. Observations on Melo’s Dictionary and Etymological Pitfalls**

Melo’s dictionary is incredibly comprehensive, yet it presents a few quirks and pitfalls.

On the one hand, the word table has a clear genealogy, but it is truncated in the dictionary. This case illustrates a key limitation of the resource: the etymological chain `eng: table → enm: table` omits deeper roots––table derives from Old English/Germanic *tabal*, itself from Old High German *zabel*, ultimately tracing back to Latin *tabula*. Interestingly, the semantic continuity dominates over morphological lineage—by c. 1300, the word referred to "a piece of furniture consisting of a flat top on legs." The more common Latin word for this was *mensa* (see [etymonline](https://www.etymonline.com/search?q=table)), while Old English writers preferred bord. Nonetheless, tracing semantic shifts is another odyssey in itself.

On the other hand, *warrior* presents no immediate etymology because it is a derived form of *war*, which itself comes from Middle English *werre*. Melo’s dictionary contains this information, but navigating it can be complex:

<figure style="text-align: center;">
    <img src="./assets/ety_CLI_warrior.png" alt="Ety_Warrior" width="400"/>
    <figcaption><em>Etymology of <strong>Warrior</strong> via CLI ety</em></figcaption>
</figure>

One major risk is falling into infinite recursion when parsing the full etymological chain. Not all entries break cleanly at rel:etymological_origin_of, and some may include multiple recursive or overlapping relation types. It’s worth remembering that the dictionary contains over 6.03 million entries and occupies nearly 1 GB in memory.

Another issue is suffix-chasing, as shown in the output for *ety* `goddess -r`:

<figure style="text-align: center;">
    <img src="./assets/ety_CLI_goddess.png" alt="Ety_Goddess" width="400"/>
    <figcaption><em>Etymology of <strong>Goddess</strong> via CLI ety</em></figcaption>
</figure>


The blatant oversight is, however, words like *hero*, which etymology isn't that obscure just ommited. 

In [54]:
hero_etym = etymology_df[etymology_df["word"] == "eng: hero"]
print(hero_etym)

       word       relation                    target_word      
795625  eng: hero  rel:etymological_origin_of    eng: anti-hero
795626  eng: hero  rel:etymological_origin_of     eng: antihero
795627  eng: hero  rel:etymological_origin_of    eng: cyberhero
795628  eng: hero  rel:etymological_origin_of      eng: heroess
795629  eng: hero  rel:etymological_origin_of     eng: heroical
795630  eng: hero  rel:etymological_origin_of     eng: herolike
795631  eng: hero  rel:etymological_origin_of     eng: heroship
795632  eng: hero  rel:etymological_origin_of     eng: megahero
795633  eng: hero  rel:etymological_origin_of      eng: nonhero
795634  eng: hero  rel:etymological_origin_of        eng: shero
795635  eng: hero  rel:etymological_origin_of    eng: superhero
795636  eng: hero  rel:etymologically_related       eng: heroic
795637  eng: hero  rel:etymologically_related      eng: heroics
795638  eng: hero  rel:etymologically_related      eng: heroine
795639  eng: hero  rel:etymologically_re

___ 
### **Step 2. Building an etymologycal lexicon**

In [55]:
# ----------------------------------------------------------------------
# 1. Unique words in Odysseys
# ----------------------------------------------------------------------

tokens_set = set().union(*df['tokens'])
etymolo_list_ody = list(tokens_set)
etymolo_list_ody.sort()
print(len(etymolo_list_ody))
print(etymolo_list_ody[:11])

20543
['aback', 'abandon', 'abandoned', 'abandoning', 'abased', 'abashed', 'abated', 'abating', 'abc', 'abeam', 'abed']
time: 31.9 ms (started: 2025-04-21 15:39:50 +02:00)


In [56]:
#----------------------------------------------------------------------
# 2. polishing our function
#----------------------------------------------------------------------

def get_etymology(word, df):
    etymology_chain = [word]  # Store the lineage of words
    
    while True:
        # Find the row where 'word' matches
        etymology_row = df[df["word"] == word]
        
        # If no match is found, stop the loop
        if etymology_row.empty:
            break
        
        # Get the next word in the etymological chain
        next_word = etymology_row["target_word"].values[-1] # 
        
        # Append to the chain and set up for the next iteration
        etymology_chain.append(next_word)
        word = next_word  # Set the next search target

    return etymology_chain[-1:]

#----------------------------------------------------------------------
# Testing with toy list
#----------------------------------------------------------------------

toy_list = ['spear', 'muse', 'shield', 'abandoning', 'abeam', 'warrior', 'goddess', 'hero']
toy_ety = []
for word in toy_list:
    etymology_chain = get_etymology(f"eng: {word}", relation_etymology_df)  # Store result here
    toy_ety.append(etymology_chain)  # Append to the list   
    print(f'The word "{word}" traces back through: {etymology_chain}.')
    print()

The word "spear" traces back through: ['ang: spere'].

The word "muse" traces back through: ['grc: Μοῦσα'].

The word "shield" traces back through: ['ang: scield'].

The word "abandoning" traces back through: ['eng: abandoning'].

The word "abeam" traces back through: ['ang: byme'].

The word "warrior" traces back through: ['eng: warrior'].

The word "goddess" traces back through: ['eng: god'].

The word "hero" traces back through: ['eng: hero'].

time: 422 ms (started: 2025-04-21 15:39:50 +02:00)


Now, our function chooses the real root of "goddess", "god". Unfortunately, "god" doesn't have a root in Melo's dictionary, however, it comes from the Old High German *got* (Gothic *guþ*). This is dissapointing, especially for a word with so rich and complex etymology: "the underlying Germanic base has been explained as a derivative (with the Germanic base of ‑ed suffix1) of the zero-grade of either of two possible Indo-European verbal bases" [(*OED*, God, N.)](https://www.oed.com/dictionary/god_n?tab=etymology).

In [57]:
god_etym = etymology_df[etymology_df["word"] == "eng: god"]
print(god_etym)

       word      relation                    target_word          
768225  eng: god  rel:etymological_origin_of            eng: begod
768226  eng: god  rel:etymological_origin_of         eng: godchild
768227  eng: god  rel:etymological_origin_of          eng: goddess
768228  eng: god  rel:etymological_origin_of        eng: godfather
768229  eng: god  rel:etymological_origin_of          eng: godless
768230  eng: god  rel:etymological_origin_of          eng: godlike
768231  eng: god  rel:etymological_origin_of          eng: godling
768232  eng: god  rel:etymological_origin_of          eng: godlore
768233  eng: god  rel:etymological_origin_of          eng: godmama
768234  eng: god  rel:etymological_origin_of          eng: godpapa
768235  eng: god  rel:etymological_origin_of          eng: godship
768236  eng: god  rel:etymological_origin_of           eng: nongod
768237  eng: god  rel:etymological_origin_of         eng: undergod
768238  eng: god  rel:etymological_origin_of            eng: u

In [58]:
# ----------------------------------------------------------------------
# Step 3.1 Etymology DataFrame Testing
# ----------------------------------------------------------------------

len(toy_list), len(toy_ety)

etymology_df = pd.DataFrame(toy_list, columns=["word"])
etymology_df["etymology"] = toy_ety
etymology_df.T

,0,1,2,3,4,5,6,7
word,spear,muse,shield,abandoning,abeam,warrior,goddess,hero
etymology,[ang: spere],[grc: Μοῦσα],[ang: scield],[eng: abandoning],[ang: byme],[eng: warrior],[eng: god],[eng: hero]


time: 92.5 ms (started: 2025-04-21 15:39:51 +02:00)


In [59]:
# ----------------------------------------------------------------------
# Step 3.2: Etymology DataFrame Construction
# ----------------------------------------------------------------------
# This step extracts the etymological root of each word in the Odyssey-related
# vocabulary list (`etymolo_list_ody`) by tracing the etymology chain using
# the `relation_etymology_df` DataFrame.

# The process involves:
# 1. Defining a cycle-aware `get_etymology` function that recursively follows
#    'target_word' links in the etymological dictionary, with cycle detection
#    to prevent infinite loops.
# 2. Iterating through each word in `etymolo_list_ody`, retrieving the final 
#    etymological form (deepest known origin), and storing it.
# 3. Constructing a new DataFrame `etymology_df` with the format:
#    | word | etymology |
# ----------------------------------------------------------------------

# Counter for cycle warnings
cycle_warning_count = 0

# 1. Optimized get_etymology function with cycle detection
def get_etymology(word, df):
    global cycle_warning_count  # Use global to modify the outer counter
    etymology_chain = [word]
    visited_words = set()

    while True:
        if word in visited_words:
            print(f"Warning: Cycle detected for '{word}'. Stopping to prevent infinite loop.")
            cycle_warning_count += 1
            break
        visited_words.add(word)

        etymology_row = df[df["word"] == word]
        if etymology_row.empty:
            break

        try:
            next_word = etymology_row["target_word"].values[-1]
            etymology_chain.append(next_word)
            word = next_word
        except Exception as e:
            print(f"Error processing word '{word}': {e}")
            break

    return etymology_chain[-1:]
    # return etymology_chain[-1:], cycle_detected # alt return for modularity

# 2. Process all words
ety_list = []

for i, word in enumerate(etymolo_list_ody, start=1):  
    try:
        etymology_chain = get_etymology(f"eng: {word}", relation_etymology_df)
        ety_list.append(etymology_chain)  
    except Exception as e:
        print(f"Skipping '{word}' due to error: {e}")
        ety_list.append([])

    if i % 500 == 0:
        print(f"Processed {i} words...")

print("Processing complete!")
print(f"Number of 'Warning: Cycle detected': {cycle_warning_count}")

# 3. Create the DataFrame
etymology_df = pd.DataFrame({"word": etymolo_list_ody, "etymology": ety_list})

# 4. Check a sample of the DataFrame
print(etymology_df.sample())

Processed 500 words...
Processed 1000 words...
Processed 1500 words...
Processed 2000 words...
Processed 2500 words...
Processed 3000 words...
Processed 3500 words...
Processed 4000 words...
Processed 4500 words...
Processed 5000 words...
Processed 5500 words...
Processed 6000 words...
Processed 6500 words...
Processed 7000 words...
Processed 7500 words...
Processed 8000 words...
Processed 8500 words...
Processed 9000 words...
Processed 9500 words...
Processed 10000 words...
Processed 10500 words...
Processed 11000 words...
Processed 11500 words...
Processed 12000 words...
Processed 12500 words...
Processed 13000 words...
Processed 13500 words...
Processed 14000 words...
Processed 14500 words...
Processed 15000 words...
Processed 15500 words...
Processed 16000 words...
Processed 16500 words...
Processed 17000 words...
Processed 17500 words...
Processed 18000 words...
Processed 18500 words...
Processed 19000 words...
Processed 19500 words...
Processed 20000 words...
Processed 20500 word

In [69]:
# ----------------------------------------------------------------------
# Percentage of words missing etymology
# ----------------------------------------------------------------------
warnings = cycle_warning_count 
total_words = len(etymolo_list_ody)
warning_percentage = (warnings / total_words) * 100
print(f"Percentage of warnings: {warning_percentage:.2f}%")

Percentage of warnings: 0.12%
time: 1.55 ms (started: 2025-04-21 16:01:01 +02:00)


In [ ]:
# ----------------------------------------------------------------------
# Milestone: Etymology DataFrame
# ----------------------------------------------------------------------
e.check_df(etymology_df)

# Save the DataFrame as a TSV file
etymology_df.to_csv('/Users/debr/odysseys_en/word_ety_odysseys.tsv',
                     sep='\t', index=False, encoding='utf-8')

Mr righteous here has no missing values!

* df columns: Index(['word', 'etymology'], dtype='object') 

* Shape: (20543, 2) 

* Total memory in MB: 2.807876
time: 25.7 ms (started: 2025-04-21 15:22:35 +02:00)


## Word-to-Etymology Map

Goal: Getting etymological labels into odysseys_df

In [81]:
# Step 1. Explode tokens column

odysseys_df = df_bkp.copy()
etymologypath = '/Users/debr/odysseys_en/word_ety_odysseys.tsv'

etymology_df = pd.read_csv(etymologypath, sep="\t", names=["word", "etymology"], encoding="utf-8")
etymology_df['etymology'] = etymology_df['etymology'].str.extract(r"\['(\w{3})").astype("category")

etymology_df.sample(8, random_state=42)

,word,etymology
12507,scopes,eng
1119,bat,fro
15537,underrate,eng
16693,woodpile,ang
13855,stain,enm
361,aldermen,eng
14676,tease,ang
12949,shifts,eng


time: 43.4 ms (started: 2025-04-21 16:07:03 +02:00)


In [ ]:
# ----------------------------------------------------------------------
# Labelling the etymology of every word in all our Odysseys
# ----------------------------------------------------------------------

# Step 1. Explode tokens column 
odyssey_exploded = odysseys_df.explode("tokens")

# Merge with etymology_df to get etymology
odyssey_exploded = odyssey_exploded\
                            .merge(etymology_df[['word', 'etymology']], 
                            left_on='tokens', right_on='word', how='left')

# Group back by translator and book_num to form lists of etymology
odysseys_df['etymology'] = odyssey_exploded\
                            .groupby(['translator', 'book_num'],
                                      observed=True)['etymology']\
                                    .apply(list).reset_index(drop=True)

odysseys_df['etymology_counts'] = odysseys_df['etymology'].apply(lambda x: dict(Counter(x)))
odysseys_df.sample(6, random_state=42)

,translator,book_num,text,tokens,etymology,etymology_counts
137,Green,18,"[Now there came up a public beggar, whose custom it was to beg\n, through the town of Ithake, well known for his ravenous belly,\n, forever guzzling and swilling. There was no strength in him,\n, ...","[came, public, beggar, whose, custom, beg, town, ithake, well, known, ravenous, belly, forever, guzzling, swilling, strength, force, yet, great, bulk, made, imposing, sight, name, arnaios, lady, m...","[eng, ang, eng, eng, eng, eng, eng, eng, eng, eng, enm, eng, non, eng, eng, ang, ang, ang, eng, eng, ang, eng, eng, eng, fro, ang, eng, lat, eng, enm, eng, heb, ang, enm, ang, heb, eng, eng, eng, ...","{'eng': 836, 'ang': 351, 'enm': 275, 'non': 29, 'fro': 45, 'lat': 99, 'heb': 7, 'xno': 9, 'san': 2, 'fra': 15, 'grc': 8, 'p_g': 2, 'cym': 1, 'frm': 1, 'ara': 1, 'tur': 1, 'ita': 2, 'spa': 1}"
30,Fitzgerald,7,"[As Lord Odysseus prayed there in the grove \n, the girl rode on, behind her strapping team, \n, and came late to the mansion of her father, \n, where she reined in at the courtyard gate. Her brot...","[lord, odysseus, prayed, grove, girl, rode, behind, strapping, team, came, late, mansion, father, reined, courtyard, gate, brothers, awaited, like, tall, gods, court, circling, lead, mules, away, ...","[eng, ang, enm, eng, eng, ang, eng, eng, eng, eng, ang, lat, eng, ang, lat, eng, eng, eng, eng, eng, fro, eng, ang, eng, eng, eng, ang, eng, enm, eng, eng, ang, ang, ang, ang, enm, enm, ang, eng, ...","{'eng': 921, 'ang': 408, 'enm': 270, 'lat': 107, 'fro': 45, 'frm': 7, 'xno': 14, 'non': 21, 'grc': 17, 'msa': 1, 'cym': 1, 'fra': 7, 'p_g': 2, 'goh': 1, 'tur': 1, 'fin': 2, 'ita': 1, 'heb': 1, 'po..."
119,Wilson,24,"[Then Hermes called the spirits of the suitors\n, out of the house. He held the golden wand\n, with which he casts a spell to close men’s eyes\n, or open those of sleepers when he wants.\n, He led...","[hermes, called, spirits, suitors, house, held, golden, wand, casts, spell, close, men, eyes, open, sleepers, wants, led, spirits, followed, squeaking, like, bats, secret, crannies, cave, cling, t...","[eng, eng, eng, eng, eng, enm, ang, eng, eng, eng, enm, ang, eng, eng, eng, eng, eng, eng, enm, eng, eng, eng, eng, eng, eng, eng, eng, enm, ang, lat, eng, eng, eng, ang, fro, eng, eng, eng, eng, ...","{'eng': 1382, 'enm': 396, 'ang': 688, 'lat': 144, 'fro': 70, 'grc': 19, 'arg': 1, 'xno': 17, 'fra': 17, 'ara': 1, 'non': 37, 'cym': 5, nan: 2, 'p_g': 2, 'msa': 1, 'frm': 3, 'gml': 1, 'heb': 1, 'it..."
29,Fitzgerald,6,"[Far gone in weariness, in oblivion, \n, the noble and enduring man slept on; \n, but Athena in the night went down the land \n, of the Phaiakians, entering their city. \n, In days gone by, these ...","[far, gone, weariness, oblivion, noble, enduring, man, slept, athena, night, went, land, phaiakians, entering, city, days, gone, men, held, hypereia, country, wide, dancing, grounds, near, overbea...","[fro, ang, eng, ang, eng, eng, ang, eng, enm, ang, ang, eng, eng, fro, eng, eng, lat, eng, enm, eng, enm, eng, eng, eng, lat, eng, eng, eng, lat, eng, eng, eng, enm, eng, enm, eng, eng, enm, lat, ...","{'fro': 48, 'ang': 399, 'eng': 802, 'enm': 256, 'lat': 104, 'frm': 5, 'xno': 15, 'grc': 18, 'fra': 11, 'dum': 8, 'non': 19, nan: 2, 'tur': 1, 'por': 1, 'heb': 1, 'p_g': 1, 'cym': 1}"
142,Green,23,"[Chuckling, the old woman ascended to the upper chamber,\n, to bring her mistress the news that her dear husband was there,\n, in the house: her feet hobbled, but her knees moved briskly,\n, and s...","[chuckling, old, woman, ascended, upper, chamber, bring, mistress, news, dear, husband, house, feet, hobbled, knees, moved, briskly, stood, penelope, head, addressed, saying, wake, penelope, dear,...","[eng, ang, enm, eng, eng, eng, ang, enm, eng, ang, enm, non, enm, eng, ang, eng, eng, eng, eng, eng, eng, eng, enm, eng, ang, eng, ang, lat, ang, ang, eng, ang, enm, ang, eng, enm, ang, eng, enm, ...","{'eng': 700, 'ang': 363, 'enm': 216, 'non': 13, 'lat': 8

time: 194 ms (started: 2025-04-21 16:08:45 +02:00)
